# Brain Tumour Classification – Improved CNN

**Improvements over the original:**
1. Transfer Learning with MobileNetV2 (pre-trained on ImageNet)
2. Aggressive Data Augmentation
3. Proper Callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
4. Two-phase training: frozen base → fine-tuning
5. GlobalAveragePooling2D instead of Flatten
6. Standard 224×224 input size

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np

print(f"TensorFlow version: {tf.__version__}")

## 1. Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────
IMG_SIZE = (224, 224)          # Standard size for MobileNetV2
BATCH_SIZE = 32
NUM_CLASSES = 4
INITIAL_EPOCHS = 20            # Phase 1: train top layers only
FINE_TUNE_EPOCHS = 30          # Phase 2: fine-tune base model
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
FINE_TUNE_AT_LAYER = 100       # Unfreeze layers from this index onward

TRAIN_DIR = 'dataset/archive/Training'
TEST_DIR  = 'dataset/archive/Testing'

## 2. Data Augmentation & Generators

**Key change:** The training generator now includes rotation, shifts, zoom, flips, and brightness changes to help the model generalise better on a small dataset.

In [ ]:
# ── Training: heavy augmentation ─────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2]
)

# ── Testing: rescale ONLY (no augmentation) ──────────────
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False           # Keep order for evaluation
)

print(f"\nClasses: {train_generator.class_indices}")
print(f"Training samples:   {train_generator.samples}")
print(f"Testing samples:    {test_generator.samples}")

## 3. Visualise Augmented Samples

In [ ]:
class_names = list(train_generator.class_indices.keys())

images, labels = next(train_generator)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(class_names[np.argmax(labels[i])], fontsize=12)
    ax.axis('off')
plt.suptitle('Augmented Training Samples', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Build Model – Transfer Learning with MobileNetV2

MobileNetV2 is pre-trained on ImageNet (1.4M images, 1000 classes). We freeze its weights and add our own classification head on top.

In [ ]:
# ── Load pre-trained MobileNetV2 (without top classifier) ─
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze ALL layers in the base model
base_model.trainable = False

# ── Add custom classification head ───────────────────────
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

print(f"Total layers:     {len(model.layers)}")
print(f"Trainable params: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")
print(f"Non-trainable:    {sum(tf.keras.backend.count_params(w) for w in model.non_trainable_weights):,}")

## 5. Phase 1 – Train Top Layers (Base Frozen)

We first train only the new classification head while keeping MobileNetV2 frozen.

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("=" * 60)
print("PHASE 1: Training classification head (base model frozen)")
print("=" * 60)

history1 = model.fit(
    train_generator,
    epochs=INITIAL_EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks_phase1
)

## 6. Phase 2 – Fine-Tune Base Model

Now we unfreeze the top layers of MobileNetV2 and train with a **very low learning rate** to fine-tune the pre-trained features for our specific task.

In [ ]:
# ── Unfreeze the top layers of the base model ────────────
base_model.trainable = True

# Freeze all layers BEFORE FINE_TUNE_AT_LAYER
for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Fine-tuning {trainable_count} layers out of {len(base_model.layers)} in base model")

# ── Re-compile with a much lower learning rate ───────────
model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model_final.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

total_epochs = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

print("=" * 60)
print("PHASE 2: Fine-tuning base model with low learning rate")
print("=" * 60)

history2 = model.fit(
    train_generator,
    epochs=total_epochs,
    initial_epoch=len(history1.history['loss']),
    validation_data=test_generator,
    callbacks=callbacks_phase2
)

## 7. Evaluate on Test Set

In [ ]:
loss, accuracy = model.evaluate(test_generator)
print(f"\n{'='*40}")
print(f"  Final Test Loss:     {loss:.4f}")
print(f"  Final Test Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"{'='*40}")

## 8. Plot Training History

In [ ]:
# Combine histories from both phases
acc      = history1.history['accuracy']      + history2.history['accuracy']
val_acc  = history1.history['val_accuracy']   + history2.history['val_accuracy']
loss_h   = history1.history['loss']           + history2.history['loss']
val_loss = history1.history['val_loss']        + history2.history['val_loss']

epochs_range = range(1, len(acc) + 1)
phase1_end = len(history1.history['loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ── Accuracy ──
ax1.plot(epochs_range, acc, 'b-', label='Train Accuracy')
ax1.plot(epochs_range, val_acc, 'r-', label='Val Accuracy')
ax1.axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Training & Validation Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Loss ──
ax2.plot(epochs_range, loss_h, 'b-', label='Train Loss')
ax2.plot(epochs_range, val_loss, 'r-', label='Val Loss')
ax2.axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Training & Validation Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print('Saved training_history.png')

## 9. Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Reset generator to start
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Classification Report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion_matrix.png')

## 10. Predict on a Single Image

In [ ]:
def predict_image(image_path, model, class_names, img_size=(224, 224)):
    """Load an image, predict its class, and display the result."""
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    prediction = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100
    
    plt.figure(figsize=(6, 6))
    plt.imshow(load_img(image_path, target_size=img_size))
    plt.title(f'Predicted: {predicted_class} ({confidence:.1f}%)', fontsize=14)
    plt.axis('off')
    plt.show()
    
    print(f"Predictions: {dict(zip(class_names, prediction[0].round(4)))}")
    return predicted_class, confidence

# Example prediction
predict_image('dataset/archive/Testing/glioma/Te-gl_0010.jpg', model, class_names)

## 11. Save Final Model

In [ ]:
model.save('brain_tumour_model_final.keras')
print('Model saved to brain_tumour_model_final.keras')
print(f'Final accuracy: {accuracy*100:.1f}%')